# 🐍 Master Guide: 02: Arguments, Keyword Var-Args & Scope Resolution

> **A Complete Foundations-to-Advanced Reference Guide**  
> Covers: Parameter Hierarchy, Default Mutable Argument Trap, `*args` and `**kwargs`, LEGB Scope Resolution, `global` vs. `globals()`, and Closures.
---

## 1. The 4 Fundamental Parameter Types & Hierarchy

Python provides a strict syntactic hierarchy for function parameters. When defining a function, parameters **MUST** appear in this exact order:

$$\text{def func(} \underbrace{\text{positional}}_{1}, \quad \underbrace{\text{default=val}}_{2}, \quad \underbrace{\text{*args}}_{3}, \quad \underbrace{\text{keyword-only}}_{4}, \quad \underbrace{\text{**kwargs}}_{5} \text{):}$$

| Parameter Type | Syntax | Internal Storage | Characteristic |
|---|---|---|---|
| **1. Positional** | `a, b` | Local variables | Bound in exact order of passed arguments. |
| **2. Default** | `c=10` | Local variables | Fallback value used if argument is omitted. |
| **3. Variable-Length (`*args`)** | `*args` | **`tuple`** | Captures any number of excess positional arguments. |
| **4. Keyword-Only** | `*, key=val` | Local variables | Must be passed explicitly by keyword name. |
| **5. Keyword Var-Args (`**kwargs`)** | `**kwargs` | **`dict`** | Captures any number of excess named key=value arguments. |

In [ ]:
# The 4 Parameter Types Execution Suite
def complete_order_dispatch(order_id, customer_name, carrier="FedEx", *items, priority=False, **metadata):
    print("=" * 55)
    print("               ORDER DISPATCH SUMMARY")
    print("=" * 55)
    print(f"  Order ID               : #{order_id}")
    print(f"  Customer Name          : {customer_name}")
    print(f"  Carrier Service        : {carrier}")
    print(f"  Priority Dispatch?     : {priority}")
    print(f"  Packaged Items (*args) : {items} (Total: {len(items)})")
    print("  Metadata (**kwargs)    :")
    for key, val in metadata.items():
        print(f"    {key:<20}: {val}")
    print("=" * 55)

complete_order_dispatch(
    9021, "Vikram Sharma", "DHL Express",
    "Mechanical Keyboard", "USB-C Cable", "Ergonomic Mouse",
    priority=True,
    warehouse="Hyderabad-WH-04", weight_kg=2.45, gift_wrap=True
)

## 2. Unpacking Arguments: `*` and `**` Operators

In addition to packing arguments inside function definitions, the asterisk operators can be used by callers to **unpack existing collections into separate function arguments**:
- `*list_or_tuple`: Unpacks items as sequential positional arguments.
- `**dict`: Unpacks key-value pairs as named keyword arguments.

In [ ]:
# Argument Unpacking Demonstration
def configure_server(host, port, timeout, ssl_enabled):
    print(f"Connecting to {host}:{port} (Timeout: {timeout}s | SSL: {ssl_enabled})")

# Unpacking a list/tuple via *
conn_tuple = ("192.168.1.100", 8443, 30, True)
configure_server(*conn_tuple)

# Unpacking a dictionary via **
conn_dict = {
    "host": "api.production.internal",
    "port": 443,
    "timeout": 15,
    "ssl_enabled": True
}
configure_server(**conn_dict)

## 3. Variable Scope Resolution: The LEGB Rule

When Python encounters a variable name, it searches for that name in **4 nested scopes** in strict sequential order:

```
+--------------------------------------------------------------+
|                     LEGB SCOPE RESOLUTION                    |
|                                                              |
|   1. [L] Local Scope        --> Inside the current function  |
|               |                                              |
|               v                                              |
|   2. [E] Enclosing Scope    --> In enclosing nested functions|
|               |                                              |
|               v                                              |
|   3. [G] Global Scope       --> At the top level of module   |
|               |                                              |
|               v                                              |
|   4. [B] Built-in Scope     --> Python built-ins (len, print)|
+--------------------------------------------------------------+
```

In [ ]:
# LEGB Scope Demonstration
global_var = "G: I am Global"

def outer_enclosing():
    enclosing_var = "E: I am Enclosing"
    
    def inner_local():
        local_var = "L: I am Local"
        print(local_var)
        print(enclosing_var)
        print(global_var)
        print("B: Built-in abs(-10) =", abs(-10))
        
    inner_local()

outer_enclosing()

## 4. Modifying Outer Scope: `global` vs. `globals()`

### The Name Collision Dilemma:
When a local variable shares the **exact same name** as a global variable:
- Reading the variable name refers to the **local** variable (shadowing the global).
- `global var_name`: Informs Python that all assignments modify the module-level global variable.
- `globals()['var_name']`: Accesses and modifies the global variable dictionary directly **without shadowing the local variable**!

In [ ]:
# Global Scope Resolution & globals() Access
counter = 1000  # Module-level Global

def scope_collision_resolver():
    counter = 50  # Local variable with same name!
    print("Local counter value                  :", counter)
    print("Global counter via globals()['counter']:", globals()['counter'])
    
    # Updating global variable directly without shadowing local
    globals()['counter'] += 500
    print("Global counter after globals() update :", globals()['counter'])

scope_collision_resolver()
print("Module global counter final value     :", counter)

## 5. Tricky Gotchas & Common Pitfalls

### ⚠️ Gotcha 1: The Default Mutable Argument Trap (Classic Interview Question)
Default arguments are evaluated **ONCE when the function is defined**, NOT every time the function is called!
If you use a mutable object (`lst=[]` or `d={}`) as a default argument, **all invocations share the exact same object**!

In [ ]:
# The Default Mutable Argument Trap Demonstration
# BUGGY IMPLEMENTATION (AVOID!):
def append_to_cache(item, cache=[]):
    cache.append(item)
    return cache

print("Call 1:", append_to_cache("A"))
print("Call 2:", append_to_cache("B")) # Returns ['A', 'B'] instead of ['B']!

# CORRECT PRODUCTION FIX: Use None as sentinel!
def append_to_cache_safe(item, cache=None):
    if cache is None:
        cache = []  # Brand new list created on every invocation!
    cache.append(item)
    return cache

print("\nSafe Call 1:", append_to_cache_safe("A"))
print("Safe Call 2:", append_to_cache_safe("B"))  # Returns ['B'] as expected!

## 6. Summary & Quick Revision Cheat Sheet

| Parameter / Scope Construct | Syntax Signature | Core Engineering Rule |
|---|---|---|
| **Positional Parameters** | `def f(a, b):` | Order matters; mandatory. |
| **Default Parameters** | `def f(a, b=10):` | Must follow positional; use `None` for mutables. |
| **`*args`** | `def f(*args):` | Packs extra positionals into a `tuple`. |
| **`**kwargs`** | `def f(**kwargs):` | Packs extra named keywords into a `dict`. |
| **LEGB Hierarchy** | `Local -> Enclosing -> Global -> Built-in` | First match wins. |
| **`global` keyword** | `global x` | Rebinds module-level variable from inside function. |
| **`globals()`** | `globals()['x']` | Dict of module namespace; bypasses local shadowing. |